# Lab 02 — Ingerir um arquivo e deduplicar a reentrega (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite).

Objetivo: **ler um CSV** (arquivo 'pousado') e **deduplicar** linhas reentregues, ficando com a versão mais recente.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
# Simula um arquivo que 'pousou' na landing zone (com uma linha reentregue: id 1)
csv = '''id,valor,carregado_em
1,100,2026-08-10
2,200,2026-08-10
3,300,2026-08-11
1,150,2026-08-12
'''
open('eventos.csv','w').write(csv)
con.execute("CREATE TABLE raw_eventos AS SELECT * FROM read_csv_auto('eventos.csv')")
con.execute('SELECT * FROM raw_eventos ORDER BY carregado_em').df()

## 1. O problema: reentrega duplica
O `id=1` aparece duas vezes (o arquivo foi reprocessado). Contagem por id:

In [ ]:
con.execute('SELECT id, COUNT(*) AS versoes FROM raw_eventos GROUP BY id ORDER BY id').df()

## 2. Dedup: manter a versão mais recente por chave
`ROW_NUMBER()` numera as versões por `id` (mais nova primeiro); ficamos com `rn = 1`.

In [ ]:
con.execute('''
  SELECT id, valor
  FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY carregado_em DESC) AS rn
        FROM raw_eventos) t
  WHERE rn = 1
  ORDER BY id
''').df()

## 3. Sua vez (mini-desafio)
Traga o `valor` mais recente do `id = 1` (um número). Verifique.

In [ ]:
resposta = con.execute('''
  SELECT valor FROM raw_eventos WHERE id = 1 ORDER BY carregado_em DESC LIMIT 1
''').fetchone()[0]
resposta

In [ ]:
def verificar(v):
    try:
        assert v == 150, 'A versão mais recente do id=1 é 150 (2026-08-12).'
        print('✅ Correto! Reentrega deduplicada — ficou a versão mais nova.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)